# 13-6. 프로젝트 A — KAPE 결과 정규화 예제

## Goal

- 제공 합성 CSV를 공통 레코드로 정규화합니다.
- 누락·행 오류·시각 없는 기록을 보존합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

저장소의 `examples/13-kape-triage` 모듈과 임시 디렉터리를 사용합니다.


## Steps

### 합성 사건 생성과 정규화

실제 사건 자료 없이 제공 생성기와 파이프라인의 `load_case()`를 실행합니다.


In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
from tempfile import TemporaryDirectory


def project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "examples/13-kape-triage/pipeline.py").is_file():
            return candidate
    raise RuntimeError("저장소 루트에서 Notebook을 실행해야 합니다")


def load_module(name: str, path: Path):
    spec = spec_from_file_location(name, path)
    module = module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


MODULE_ROOT = project_root() / "examples/13-kape-triage"
sample_module = load_module("chapter13_make_sample", MODULE_ROOT / "make_sample.py")
pipeline_module = load_module("chapter13_pipeline", MODULE_ROOT / "pipeline.py")

with TemporaryDirectory() as directory:
    manifest_path = sample_module.make_sample(Path(directory) / "input")
    manifest, manifest_hash, events, issues, coverage = pipeline_module.load_case(manifest_path)
    project_a_summary = {
        "events": len(events),
        "issues": len(issues),
        "missing": sum(item["status"] == "missing" for item in coverage),
        "undated": sum(event["timestamp_utc"] is None for event in events),
        "manifest_sha256_length": len(manifest_hash),
    }
print(project_a_summary)


{'events': 14, 'issues': 1, 'missing': 1, 'undated': 1, 'manifest_sha256_length': 64}


## Checks

교안이 정한 합성 사건의 기준 건수를 확인합니다.


In [2]:
assert project_a_summary == {
    "events": 14,
    "issues": 1,
    "missing": 1,
    "undated": 1,
    "manifest_sha256_length": 64,
}
print("프로젝트 A 기준 검사 통과")


프로젝트 A 기준 검사 통과


## Next Steps

실제 KAPE·EZ Tools 출력에는 사용한 버전의 실제 헤더에 맞춘 별도 어댑터가 필요합니다.
